# B3: Generic I-JEPA Linear Probe

**Leaf-JEPA IRP** | Stage 3 — Baseline Establishment

## Rationale

B3 is one half of the **most important experiment in this project** (RQ1).
It measures the quality of **generic** (ImageNet-pretrained) I-JEPA representations
for plant disease classification.

Setup: load Meta I-JEPA ViT-H/14 checkpoint, **freeze every parameter**,
train ONLY a linear classification head. The resulting macro-F1 is compared
directly with B5 (Leaf-JEPA) — the gap IS the RQ1 answer.

Because the setup is identical between B3 and B5 (same architecture, same head,
same training, same evaluation — only the encoder weights differ), **no confounders exist**.

## Initialization

In [5]:
import sys, json, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader


PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from stage3_baseline_establishment.config_stage3 import *
from stage3_baseline_establishment.baseline_utils import (
    seed_everything, PlantVillageDataset, load_split,
    evaluate, save_results, plot_confusion_matrix, plot_tsne,
    load_ijepa_encoder, EarlyStopping
)

from stage2_dataset_preparation.outputs.augmentation.transforms import (
    get_pretrain_transform, get_eval_transform, get_finetune_transform
)

verify_config()
seed_everything(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

hyperparams = {
    "baseline": "B3",
    "model": "I-JEPA ViT-H/14 (generic, ImageNet pretrained) + Linear Head",
    "encoder_params": "632M (ALL FROZEN)",
    "head_params": "~49K (1280 x 38 + 38)",
    "checkpoint": str(IJEPA_CHECKPOINT),
    "optimiser": "AdamW (head only)",
    "lr": LP_LR,
    "batch_size": LP_BATCH_SIZE,
    "max_epochs": LP_EPOCHS,
    "patience": LP_PATIENCE,
}
BASELINES_DIR.mkdir(parents=True, exist_ok=True)
save_results(hyperparams, BASELINES_DIR / "B3_hyperparams.json")

  ✅  All config checks passed.
  Saved: /workspace/leaf-jepa/stage3_baseline_establishment/outputs/baselines/B3_hyperparams.json


## Extract and cache frozen features

In [6]:
FEATURES_SAVE_DIR = PROJECT_ROOT / "stage3_baseline_establishment/outputs/extracted_features"
FEATURES_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Define the specific file paths
TRAIN_FILE = FEATURES_SAVE_DIR / "train_features.pt"
VAL_FILE   = FEATURES_SAVE_DIR / "val_features.pt"
TEST_FILE  = FEATURES_SAVE_DIR / "test_features.pt"

# Check if all files exist
if TRAIN_FILE.exists() and VAL_FILE.exists() and TEST_FILE.exists():
    print("🚀 Extracted features found on disk. Loading...")

    train_data = torch.load(TRAIN_FILE, map_location='cpu')
    train_feats, train_labs = train_data['features'], train_data['labels']

    val_data = torch.load(VAL_FILE, map_location='cpu')
    val_feats, val_labs = val_data['features'], val_data['labels']

    test_data = torch.load(TEST_FILE, map_location='cpu')
    test_feats, test_labs = test_data['features'], test_data['labels']

else:
    print("🔍 Features not found.")
    print("Loading I-JEPA encoder...")
    encoder = load_ijepa_encoder(IJEPA_CHECKPOINT, VIT_MODEL_NAME, device)

    # Freeze ALL encoder parameters
    for param in encoder.parameters():
        param.requires_grad = False
    encoder.eval()

    n_frozen = sum(p.numel() for p in encoder.parameters())
    n_trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
    print(f"  Encoder params: {n_frozen:,} (all frozen, {n_trainable} trainable)")

    # Load splits
    train_paths, train_labels, class_names = load_split(SPLITS_DIR / "train_split.json")
    val_paths, val_labels, _ = load_split(SPLITS_DIR / "val_split.json")
    test_paths, test_labels, _ = load_split(SPLITS_DIR / "test_split.json")

    eval_tf = get_eval_transform()

    def extract_features(paths, labels, desc="Extracting"):
        ds = PlantVillageDataset(paths, labels, transform=eval_tf)
        loader = DataLoader(ds, batch_size=LP_BATCH_SIZE, shuffle=False,
                            num_workers=4, pin_memory=True)
        all_feats = []
        all_labs = []
        print(f"  {desc}: {len(ds)} images...")
        with torch.no_grad():
            for images, labs in loader:
                images = images.to(device, non_blocking=True)
                with autocast(device_type="cuda", enabled=True):
                    feats = encoder(images)
                all_feats.append(feats.float().cpu())
                all_labs.append(labs)
        return torch.cat(all_feats), torch.cat(all_labs)

    print("Starting loading Iextraction with I-JEPA encoder...")
    train_feats, train_labs = extract_features(train_paths, train_labels, "Train features")
    val_feats, val_labs = extract_features(val_paths, val_labels, "Val features")
    test_feats, test_labs = extract_features(test_paths, test_labels, "Test features")

    # Save to disk 
    torch.save({'features': train_feats, 'labels': train_labs}, TRAIN_FILE)
    torch.save({'features': val_feats, 'labels': val_labs}, VAL_FILE)
    torch.save({'features': test_feats, 'labels': test_labs}, TEST_FILE)
    print(f"✅ Features saved to {FEATURES_SAVE_DIR}")

    # Free encoder from GPU
    del encoder
    torch.cuda.empty_cache()


print("\n✅ Ready for training.")
print(f"\n  Feature dim: {train_feats.shape[1]}")
print(f"  Train: {train_feats.shape[0]}, Val: {val_feats.shape[0]}, Test: {test_feats.shape[0]}")

🔍 Features not found.
Loading I-JEPA encoder...
  Loaded checkpoint: IN1K-vit.h.14-300e.pth.tar
  Missing keys:  1
  Unexpected keys: 0
  Missing (check these): ['cls_token']
  Encoder params: 630,763,520 (all frozen, 0 trainable)
Starting loading Iextraction with I-JEPA encoder...
  Train features: 38013 images...
  Val features: 8146 images...
  Test features: 8146 images...
✅ Features saved to /workspace/leaf-jepa/stage3_baseline_establishment/outputs/extracted_features

✅ Ready for training.

  Feature dim: 1280
  Train: 38013, Val: 8146, Test: 8146


## Train linear head on cached features

In [7]:
from torch.utils.data import TensorDataset
import wandb

train_fds = TensorDataset(train_feats, train_labs)
val_fds   = TensorDataset(val_feats, val_labs)
test_fds  = TensorDataset(test_feats, test_labs)

train_floader = DataLoader(train_fds, batch_size=LP_BATCH_SIZE, shuffle=True)
val_floader   = DataLoader(val_fds, batch_size=LP_BATCH_SIZE, shuffle=False)
test_floader  = DataLoader(test_fds, batch_size=LP_BATCH_SIZE, shuffle=False)

# Linear head
head = nn.Linear(EMBED_DIM, NUM_CLASSES).to(device)
nn.init.xavier_uniform_(head.weight)
nn.init.zeros_(head.bias)

n_head_params = sum(p.numel() for p in head.parameters())
print(f"Linear head params: {n_head_params:,}")

criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.AdamW(head.parameters(), lr=LP_LR, weight_decay=LP_WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=LP_EPOCHS)
es = EarlyStopping(patience=LP_PATIENCE)

if WANDB_ENTITY:
    os.environ["WANDB__SERVICE_WAIT"] = "10"
    os.environ["WANDB_DISABLED"] = "true"
    try:
        wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                   name="B3-IJEPA-LinearProbe", reinit=True, config=hyperparams)
    except Exception:
        print("WandB init failed — training without logging")
        WANDB_ENTITY = False


print("\nTraining linear probe on cached features...")
t0 = time.time()
for epoch in range(LP_EPOCHS):
    head.train()
    running_loss = 0
    for feats_b, labs_b in train_floader:
        feats_b, labs_b = feats_b.to(device), labs_b.to(device)
        logits = head(feats_b)
        loss = criterion(logits, labs_b)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        running_loss += loss.item()
    scheduler.step()

    # Val
    head.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for feats_b, labs_b in val_floader:
            feats_b = feats_b.to(device)
            preds = head(feats_b).argmax(dim=1).cpu()
            val_preds.extend(preds.numpy())
            val_true.extend(labs_b.numpy())

    from sklearn.metrics import f1_score as sk_f1
    val_f1 = sk_f1(val_true, val_preds, average="macro", zero_division=0)

    if WANDB_ENTITY:
        wandb.log({"train_loss": running_loss / len(train_floader),
                    "val_macro_f1": val_f1, "epoch": epoch})

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d} | val_F1={val_f1:.4f}")

    # EarlyStopping on val F1 (save head state)
    if val_f1 > es.best_score + es.min_delta:
        es.best_score = val_f1
        es.best_state = {k: v.clone() for k, v in head.state_dict().items()}
        es.counter = 0
    else:
        es.counter += 1
        if es.counter >= es.patience:
            print(f"  Early stop at epoch {epoch+1}, best val F1: {es.best_score:.4f}")
            break

elapsed = time.time() - t0
if es.best_state:
    head.load_state_dict(es.best_state)

# Test evaluation
head.eval()
test_preds, test_true = [], []
with torch.no_grad():
    for feats_b, labs_b in test_floader:
        preds = head(feats_b.to(device)).argmax(dim=1).cpu()
        test_preds.extend(preds.numpy())
        test_true.extend(labs_b.numpy())

from sklearn.metrics import f1_score as sk_f1, accuracy_score as sk_acc, confusion_matrix as sk_cm

test_f1 = sk_f1(test_true, test_preds, average="macro", zero_division=0)
test_acc = sk_acc(test_true, test_preds)
per_class = sk_f1(test_true, test_preds, average=None, zero_division=0)
cm = sk_cm(test_true, test_preds, labels=list(range(NUM_CLASSES)))

print(f"\n  B3 Test macro-F1:  {test_f1:.4f}")
print(f"  B3 Test accuracy:  {test_acc:.4f}")

if WANDB_ENTITY:
    wandb.log({"test_macro_f1": test_f1, "test_accuracy": test_acc})
    wandb.finish()

b3_aggregate = {
    "baseline": "B3", "model": "I-JEPA ViT-H/14 Linear Probe (generic)",
    "macro_f1": float(test_f1), "accuracy": float(test_acc),
    "per_class_f1": [float(f) for f in per_class],
    "training_time_s": elapsed,
    "best_val_f1": float(es.best_score),
    "trainable_params": n_head_params,
    "encoder_params": n_frozen,
}
save_results(b3_aggregate, BASELINES_DIR / "B3_aggregate.json")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plot_confusion_matrix(cm, class_names,
                      FIGURES_DIR / "B3_confusion_matrix.png",
                      title=f"B3 I-JEPA Linear Probe (generic) | F1={test_f1:.3f}")

Linear head params: 48,678



Training linear probe on cached features...
  Epoch   1 | val_F1=0.8578
  Epoch  10 | val_F1=0.9717
  Epoch  20 | val_F1=0.9780
  Epoch  30 | val_F1=0.9819
  Epoch  40 | val_F1=0.9809
  Early stop at epoch 40, best val F1: 0.9819

  B3 Test macro-F1:  0.9828
  B3 Test accuracy:  0.9876


epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
test_accuracy,▁
test_macro_f1,▁
train_loss,█▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_macro_f1,▁▅▆▆▇▇▇▇▇▇▇▇█▇██████████████████████████
epoch,39
test_accuracy,0.9876
test_macro_f1,0.98275
train_loss,0.01573
val_macro_f1,0.98087


  Saved: /workspace/leaf-jepa/stage3_baseline_establishment/outputs/baselines/B3_aggregate.json
  Saved: /workspace/leaf-jepa/stage3_baseline_establishment/outputs/figures/B3_confusion_matrix.png


## t-SNE visualisation

In [8]:
test_feats_np = test_feats.numpy()
test_labs_np = np.array(test_labels)

plot_tsne(test_feats_np, test_labs_np, class_names,
          FIGURES_DIR / "B3_tsne_generic_ijepa.png",
          title="B3: Generic I-JEPA Feature Space (t-SNE)",
          perplexity=30)

print("\nB3 complete. Compare this t-SNE with B5 after Stage 4.")

  Running t-SNE on 8146 samples...
  Saved: /workspace/leaf-jepa/stage3_baseline_establishment/outputs/figures/B3_tsne_generic_ijepa.png

B3 complete. Compare this t-SNE with B5 after Stage 4.


## Critical Analysis

B3 measures generic SSL representation quality for plant disease:

1. **Expected range**: 75-88% macro-F1. Generic ImageNet features capture some disease texture but lack domain specificity.

2. **RQ1 setup**: B3 is the left side of the central comparison. The B5 - B3 gap (computed after Stage 4) is the headline result.

3. **t-SNE analysis**: Look for whether disease classes cluster by pathogen type (fungal/bacterial/viral) or by crop type. If clusters are crop-based, generic features see "plant" more than "disease" — this motivates domain pretraining.

4. **Feature caching**: Training on cached features takes minutes instead of hours, enabling rapid iteration.